# Task 3 · Training on Colab (T3-08, shared with T3-10)

Based on `common/colab_remote_gpu_setup.ipynb` (SETUP-05). Sections 1-5 set up
the session and are shared with T3-10; section 6 holds one cell per T3-08 run.

**Before running** (once):
1. **Runtime > Change runtime type > T4 GPU**.
2. The team folder `ml_tek5_checkpoints` is in your **My Drive** (or a shortcut
   to it, see the SETUP-05 notebook, section 2).
3. `ml_tek5_task3_data.zip`, made locally by
   `uv run python -m task3_uvr.pack_data`, is uploaded to
   `ml_tek5_checkpoints/task3/`.
4. A Colab secret `GITHUB_TOKEN` (fine-grained token, read-only `Contents` on
   the team repo) with notebook access on.
5. `BRANCH` below is pushed to GitHub.

Everything that must survive a disconnect goes to Drive, under
`ml_tek5_checkpoints/task3/`: checkpoints, loss logs, exported ONNX models and
the results CSVs and figures. After a disconnect, rerun sections 1-5, then the
run cell that was interrupted: it resumes from its last checkpoint (every 500
steps). A finished run cell can be rerun safely; it only exports and measures
the SDR again.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Drive

Same folder as SETUP-05. Task 3 files go to its `task3/` subfolder.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/ml_tek5_checkpoints"
if not os.path.isdir(CHECKPOINT_DIR):
    raise FileNotFoundError(
        f"{CHECKPOINT_DIR} not found: add the team folder (or a shortcut named "
        "ml_tek5_checkpoints) to My Drive first"
    )
TASK3_DIR = f"{CHECKPOINT_DIR}/task3"
SMALL_NN_DIR = f"{TASK3_DIR}/small_nn"  # T3-08 checkpoints, logs and ONNX models
DRIVE_RESULTS = f"{TASK3_DIR}/results"  # copies of results/task3_* files
for path in (TASK3_DIR, SMALL_NN_DIR, DRIVE_RESULTS):
    os.makedirs(path, exist_ok=True)
print("Task 3 folder:", TASK3_DIR)

## 3. Clone the repository

As in SETUP-05: the token is sent as a one-off header, never written to
`.git/config`. Set `BRANCH` to the branch that holds the code to run (the
T3-08 branch until it is merged, then `master`).

In [ ]:
import base64
import getpass

BRANCH = "27-t3-08-train-a-small-network-from-scratch"

try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001 - no secret, no access or not Colab: ask below
    token = None
if not token:
    token = getpass.getpass("GitHub token (read access to the team repo): ")

REPO_URL = "https://github.com/NattanCochet/G-AIA-900-PAR-9-1-machinelearning-11.git"
REPO_DIR = "/content/ml-tek5-project"
basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
AUTH_HEADER = f"AUTHORIZATION: basic {basic_auth}"

# A second Authorization header (Colab's or an earlier run's) makes GitHub
# answer 400 "Duplicate header", so clear them first.
!git config --global --unset-all http.https://github.com/.extraheader 2>/dev/null
!git config --system --unset-all http.https://github.com/.extraheader 2>/dev/null

if not os.path.isdir(REPO_DIR):
    !git -c credential.helper= -c http.extraheader="{AUTH_HEADER}" clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} -c credential.helper= -c http.extraheader="{AUTH_HEADER}" fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} -c credential.helper= -c http.extraheader="{AUTH_HEADER}" pull --ff-only origin {BRANCH}
!git -C {REPO_DIR} log --oneline -1

## 4. Dependencies

As in SETUP-05: Colab's CUDA PyTorch is kept (`uv sync` would install the CPU
wheel pinned for the laptops); only the task 3 packages Colab lacks are
installed. Keep this list in sync with `pyproject.toml`.

In [ ]:
!pip install -q soundfile onnx onnxruntime onnx2pytorch mir_eval

In [ ]:
import importlib
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import torch

assert torch.cuda.is_available(), "GPU not enabled: Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 5. Data

The zip is copied from Drive to the local disk and unzipped to
`/content/data` (reading WAV files from Drive is slow). This takes a few
minutes and is skipped when the data is already there.

In [ ]:
import shutil
import subprocess

DATA_ZIP = f"{TASK3_DIR}/ml_tek5_task3_data.zip"
DATA_DIR = "/content/data"
MUSDB7_DIR = f"{DATA_DIR}/musdb7"

if not os.path.isdir(f"{DATA_DIR}/musdb7_narrowband/test"):
    if not os.path.isfile(DATA_ZIP):
        raise FileNotFoundError(f"Upload ml_tek5_task3_data.zip to {TASK3_DIR} first")
    os.makedirs(DATA_DIR, exist_ok=True)
    shutil.copy(DATA_ZIP, "/content/data.zip")
    subprocess.run(["unzip", "-q", "-o", "/content/data.zip", "-d", DATA_DIR], check=True)
    os.remove("/content/data.zip")

os.environ["ML_TEK5_MUSDB7_DIR"] = MUSDB7_DIR
for dataset in ("musdb7", "musdb7_phone", "musdb7_narrowband"):
    counts = {s: len(os.listdir(f"{DATA_DIR}/{dataset}/{s}")) for s in ("train", "test")}
    print(dataset, counts)  # expected: 94 train, 50 test

### Results on Drive

The training script adds rows to `results/task3_sdr_clips.csv` and
`results/task3_uvr.csv` in the clone, which a disconnect wipes. Before each
run, the Drive copies (if any) replace the clone's; after it, every
`results/task3_*` file is copied back to Drive. Drive therefore holds the full
history across sessions.

In [ ]:
import glob

RESULT_CSVS = ("task3_sdr_clips.csv", "task3_uvr.csv")


def pull_results():
    for name in RESULT_CSVS:
        if os.path.exists(f"{DRIVE_RESULTS}/{name}"):
            shutil.copy(f"{DRIVE_RESULTS}/{name}", f"{REPO_DIR}/results/{name}")


def push_results():
    for path in glob.glob(f"{REPO_DIR}/results/task3_*"):
        shutil.copy(path, DRIVE_RESULTS)
    print("results copied to", DRIVE_RESULTS)

## 6. T3-08: small network from scratch

Four runs: {musdb7, musdb7_phone} x {remix, no remix}. Each one trains
`STEPS` steps of batch 8, saves its checkpoint and loss log to
`task3/small_nn/` on Drive every 500 steps, exports the ONNX model and its UVR
settings there, and measures the validation and test SDR (test for reporting
only). `--resume` is always passed: a run with no checkpoint starts from
step 0.

To train longer after a run has finished, raise `STEPS` and rerun its cell:
it resumes from the final checkpoint.

In [ ]:
STEPS = 4000
BATCH_SIZE = 8


def python_module(*args):
    """Run `python -m ...` from the repository root, print its output as it
    comes, raise if it fails."""
    cmd = [sys.executable, "-u", "-m", *map(str, args)]
    with subprocess.Popen(
        cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


def small_nn(dataset, remix):
    name = f"SmallMaskNet_{dataset}_{'remix' if remix else 'noremix'}"
    pull_results()
    try:
        python_module(
            "task3_uvr.train_small_nn",
            "--dataset", dataset,
            "--remix" if remix else "--no-remix",
            "--steps", STEPS,
            "--batch-size", BATCH_SIZE,
            "--checkpoint", f"{SMALL_NN_DIR}/{name}.pt",
            "--artifacts-dir", SMALL_NN_DIR,
            "--device", "cuda",
            "--resume",
        )
    finally:
        push_results()

### musdb7, remix

In [ ]:
small_nn("musdb7", remix=True)

### musdb7, no remix

In [ ]:
small_nn("musdb7", remix=False)

### musdb7_phone, remix

In [ ]:
small_nn("musdb7_phone", remix=True)

### musdb7_phone, no remix

In [ ]:
small_nn("musdb7_phone", remix=False)

### Optional: narrowband

Only if time is left: the hardest degradation (T3-06).

In [ ]:
# small_nn("musdb7_narrowband", remix=True)

### All loss curves

One figure with the training and validation losses of every finished run.

In [ ]:
logs = sorted(glob.glob(f"{SMALL_NN_DIR}/SmallMaskNet_*_log.csv"))
print("\n".join(logs))
python_module(
    "task3_uvr.plot_small_nn", *logs, "--output", "results/task3_small_nn_losses.png"
)
push_results()

from IPython.display import Image, display

display(Image(f"{REPO_DIR}/results/task3_small_nn_losses.png"))

## 7. Bring the results home

On Drive, `ml_tek5_checkpoints/task3/`:
- `results/`: `task3_sdr_clips.csv`, `task3_uvr.csv` and the loss figures.
  Copy them into the local `results/` folder and commit them.
- `small_nn/`: checkpoints (`.pt`), loss logs (`_log.csv`), ONNX models,
  their `_io.npz` and `model_data/<hash>.json`. These are not committed; the
  ONNX files and JSONs go to UVR for T3-09.

In [ ]:
!ls -la {DRIVE_RESULTS} {SMALL_NN_DIR}

## 8. T3-10: fine-tuning on degraded audio

Sections 1-5 are shared: rerun them, then add the T3-10 cells here, with
checkpoints under `TASK3_DIR` (for example `f"{TASK3_DIR}/finetune"`).